# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [1]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5126 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [28]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
       
        for arg in args:
            if isinstance(arg, list):
                print(f"Długość listy: {len(arg)}")
        
        
        for value in kwargs.values():
            if isinstance(value, list):
                print(f"Długość listy: {len(value)}")
        
        
        return func(*args, **kwargs)
    
    return wrapper



@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [27]:
import functools
from datetime import datetime
import time

def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            
            start_time = time.time()
            
        
            result = func(*args, **kwargs)
            
           
            end_time = time.time()
            execution_time = end_time - start_time
            
          
            current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
          
            log_entry = f"{current_datetime} | Funkcja: {func.__name__} | Czas wykonania: {execution_time:.6f}s\n"
            
           
            with open(filename, 'a', encoding='utf-8') as log_file:
                log_file.write(log_entry)
            
            return result
        
        return wrapper
    return decorator



@logger("app.log")
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")
    time.sleep(0.5) 
    return len(data_list)

@logger("operations.log")
def calculate_sum(numbers):
    print("Obliczanie sumy...")
    time.sleep(0.2)
    return sum(numbers)

@logger("app.log")
def quick_function():
    print("Szybka funkcja")
    return "Done"

--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [7]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [26]:
import logging
from datetime import datetime


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

class AccessLogger:
   
    
    def __init__(self, name):
        self.name = name
        self.private_name = f'_{name}'
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        
        value = getattr(instance, self.private_name, None)
        logging.info(f"ODCZYT atrybutu '{self.name}' | Wartość: {value}")
        return value
    
    def __set__(self, instance, value):
        old_value = getattr(instance, self.private_name, None)
        setattr(instance, self.private_name, value)
        logging.info(f"ZAPIS atrybutu '{self.name}' | Stara wartość: {old_value} | Nowa wartość: {value}")


class Uzytkownik:
 
    
    imie = AccessLogger('imie')
    wiek = AccessLogger('wiek')
    
    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek
    
    def __repr__(self):
      
        imie = getattr(self, '_imie', 'Brak')
        wiek = getattr(self, '_wiek', 'Brak')
        return f"Uzytkownik(imie='{imie}', wiek={wiek})"

--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [12]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [25]:
def collatz_generator(n):
    
    if n < 1:
        raise ValueError("Liczba początkowa musi być liczbą naturalną (n >= 1)")
    
  
    yield n
    
    
    while n != 1:
        if n % 2 == 0:
            
            n = n // 2
        else:
          
            n = 3 * n + 1
        
        yield n

---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [24]:
import functools

current_user = {"username": "admin", "role": "superuser"}

def require_role(required_role):
    
    
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
           
            if 'role' not in current_user:
                raise PermissionError(f"Brak informacji o roli użytkownika")
            
          
            if current_user['role'] != required_role:
                username = current_user.get('username', 'Nieznany')
                raise PermissionError(
                    f"Brak uprawnień! Użytkownik '{username}' ma rolę '{current_user['role']}', "
                    f"wymagana rola: '{required_role}'"
                )
            
           
            return func(*args, **kwargs)
        
        return wrapper
    return decorator



@require_role("superuser")
def delete_database():
    return "Baza danych została usunięta!"

@require_role("admin")
def manage_users():
    return "Zarządzanie użytkownikami"

@require_role("user")
def view_profile():
    return "Wyświetlanie profilu"

@require_role("moderator")
def moderate_content():
    return "Moderowanie treści"





### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [23]:
class Typed:
  
    
    def __init__(self, name, expected_type):
       
        self.name = name
        self.expected_type = expected_type
        self.private_name = f'_{name}'
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name, None)
    
    def __set__(self, instance, value):
        # Sprawdź czy wartość jest odpowiedniego typu
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"Atrybut '{self.name}' musi być typu {self.expected_type.__name__}, "
                f"otrzymano {type(value).__name__} (wartość: {value})"
            )
        setattr(instance, self.private_name, value)
    
    def __set_name__(self, owner, name):
       
        self.name = name
        self.private_name = f'_{name}'


class Person:
   
    
    name = Typed('name', str)
    age = Typed('age', int)
    salary = Typed('salary', float)
    is_active = Typed('is_active', bool)
    
    def __init__(self, name, age, salary, is_active=True):
        self.name = name
        self.age = age
        self.salary = salary
        self.is_active = is_active
    
    def __repr__(self):
        return (f"Person(name='{self.name}', age={self.age}, "
                f"salary={self.salary}, is_active={self.is_active})")





### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [22]:
def prime_generator():
   
    def is_prime(n):
       
        if n < 2:
            return False
        if n == 2:
            return True
        if n % 2 == 0:
            return False
        
  
        i = 3
        while i * i <= n:
            if n % i == 0:
                return False
            i += 2
        return True
    
   
    n = 2
    while True:
        if is_prime(n):
            yield n
        n += 1




print("\n" + "="*70)
print("TEST: Liczby pierwsze kończące się cyfrą 7")
print("="*70)

primes_ending_7 = (p for p in prime_generator() if p % 10 == 7)


primes_7_list = []
for _ in range(15):
    primes_7_list.append(next(primes_ending_7))

print(f"Pierwsze 15 liczb pierwszych kończących się na 7:")
print(primes_7_list)

print("\n" + "="*70)



TEST: Liczby pierwsze kończące się cyfrą 7
Pierwsze 15 liczb pierwszych kończących się na 7:
[7, 17, 37, 47, 67, 97, 107, 127, 137, 157, 167, 197, 227, 257, 277]

